In [ ]:
# 1
from datasets import load_dataset

dataset = load_dataset("csv", data_files="../data/train.csv")

def combine_text_fn(example):
    return {"combined_text": f"{example['prompt']} {example['A']}"}

dataset = dataset.map(combine_text_fn)
length_51 = len(dataset[51]["combined_text"])
length_51

614


In [ ]:
# 2
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
vocab_size = tokenizer.vocab_size
vocab_size

30522


In [ ]:
sep_id = tokenizer.convert_tokens_to_ids("[SEP]")
sep_id

102


In [ ]:
tokenized_outputs = tokenizer(
    list(dataset["prompt"]),
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)
shape_input_ids = tokenized_outputs["input_ids"].shape
shape_input_ids

torch.Size([2000, 128])


In [ ]:
hidden_size = 768
num_attention_heads = 12
head_dim = hidden_size // num_attention_heads
head_dim

64


In [ ]:
import torch
from transformers import AutoModel

model = AutoModel.from_pretrained("bert-base-uncased")
inputs_0 = tokenizer(dataset[0]["prompt"], return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs_0)

lh_shape = outputs.last_hidden_state.shape
lh_shape

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2050.10it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])


In [ ]:
cls_embedding = outputs.last_hidden_state[0, 0, :]
cls_sum = torch.sum(cls_embedding[:5]).item()
round(cls_sum, 4)

-1.2001


In [ ]:
model_att = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
text = "Light-ion fusion is a technique."
inputs_text = tokenizer(text, return_tensors="pt")

tokens = tokenizer.convert_ids_to_tokens(inputs_text["input_ids"][0])
fusion_index = tokens.index("fusion")

with torch.no_grad():
    outputs_att = model_att(**inputs_text)

attention_matrix = outputs_att.attentions[-1][0, 0]
weight = attention_matrix[0, fusion_index].item()
round(weight, 4)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7787.74it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.1025


In [ ]:
from sentence_transformers import SentenceTransformer, util

st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
prompt_emb = st_model.encode(dataset[0]["prompt"], convert_to_tensor=True)
option_b_emb = st_model.encode(dataset[0]["B"], convert_to_tensor=True)

similarity = util.cos_sim(prompt_emb, option_b_emb).item()
round(similarity, 4)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2440.09it/s]


0.7658


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer, util
import numpy as np

options = ["A", "B", "C"]
true_labels = dataset["answer"]

preds_tfidf = []
preds_minilm = []

st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

for row in dataset:
    row_options = [row[opt] for opt in options]
    
    tfidf_vec = TfidfVectorizer().fit_transform([row["prompt"]] + row_options)
    tfidf_sims = (tfidf_vec[0] * tfidf_vec[1:].T).toarray()[0]
    preds_tfidf.append([options[i] for i in np.argsort(tfidf_sims)[::-1]])
    
    p_emb = st_model.encode(row["prompt"], convert_to_tensor=True)
    o_embs = st_model.encode(row_options, convert_to_tensor=True)
    minilm_sims = util.cos_sim(p_emb, o_embs)[0].cpu().numpy()
    preds_minilm.append([options[i] for i in np.argsort(minilm_sims)[::-1]])

def apk(actual, predicted, k=3):
    if len(predicted) > k: predicted = predicted[:k]
    score = 0.0
    num_hits = 0.0
    for i, p in enumerate(predicted):
        if p == actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
    return score

map3_score = sum([apk(a, p) for a, p in zip(true_labels, preds_minilm)]) / len(true_labels)
count_diff = sum([1 for a, p_tf, p_mi in zip(true_labels, preds_tfidf, preds_minilm) if a not in p_tf[:3] and a in p_mi[:3]])

map3_score, count_diff

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7356.17it/s]


0.4275833333333333
0


In [ ]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
row_1 = dataset[1]
candidate_labels = [row_1["A"], row_1["B"], row_1["C"]]

res_single = classifier(row_1["prompt"], candidate_labels, multi_label=False)
top_score = res_single["scores"][0]
round(top_score, 4)

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 767.09it/s]


0.4575


In [ ]:
res_multi = classifier(row_1["prompt"], candidate_labels, multi_label=True)
sum_single = sum(res_single["scores"])
sum_multi = sum(res_multi["scores"])
abs_diff = abs(sum_single - sum_multi)
abs_diff

0.9994904224240599


In [ ]:
from transformers import pipeline

# Fix: Changed "text2text-generation" to "text-generation"
gen_pipeline = pipeline("text-generation", model="google/flan-t5-small")

row_0 = dataset[0]
prompt_str = f"Question: {row_0['prompt']}. Is the correct answer A: {row_0['A']} or B: {row_0['B']}? Answer with just the letter A or B."

output = gen_pipeline(prompt_str, max_new_tokens=5)
output[0]["generated_text"]

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 5962.01it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadMo

Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B.
